# 1 · make — Top7 contact-map data

Folds one protein with one checkpoint and stores the raw prediction, so
[`1_plot_top7_heatmap.ipynb`](1_plot_top7_heatmap.ipynb) can draw it without a GPU.

Default protein is **Top7** (`1qys_A`), the de novo design from Kuhlman et al. 2003. It lives in
the legacy 554-protein universe, not in the FoldBench monomer sets.

The recipe is [#82](https://github.com/Open-Athena/MarinFold/issues/82)'s settled
**rollout + resample**: `N_ROLLOUTS` different realizations of the same protein's document
(contacts-v1 randomizes the N-terminal offset and the statement order), one sampled contact
section from each, votes counted per residue pair, ties broken by the pairwise log-probability.

**Needs a GPU.** Everything else in this directory runs on CPU.

In [1]:
# Run from anywhere: figlib lives next to this notebook.
import sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "figlib.py").exists() else Path("notebooks/figures")
sys.path.insert(0, str(HERE.resolve()))
import figlib

In [2]:
# --- parameters -------------------------------------------------------------------------------
DATASET = "1_top7_heatmap"
PROTEIN = "denovo_pdb__1qys_A"   # dataset__stem in the legacy 554
MODEL = "contacts-v1-exp232-m2-p06-1.5B"

N_ROLLOUTS = 100                 # exp82's settled count
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = -1                       # disabled, as in the published harness
BACKEND = "auto"                 # auto -> vllm at compute capability >= 8.0, else transformers
DTYPE = "auto"                   # auto -> bfloat16; float16 overflows these weights (see README)
GPU_MEMORY_UTILIZATION = 0.85    # vLLM only; lower it if several engines share one card

PARAMETERS = dict(protein=PROTEIN, model=MODEL, n_rollouts=N_ROLLOUTS, temperature=TEMPERATURE,
                  top_p=TOP_P, top_k=TOP_K, backend=BACKEND, dtype=DTYPE,
                  gpu_memory_utilization=GPU_MEMORY_UTILIZATION)
PARAMETERS

{'protein': 'denovo_pdb__1qys_A',
 'model': 'contacts-v1-exp232-m2-p06-1.5B',
 'n_rollouts': 100,
 'temperature': 1.0,
 'top_p': 0.95,
 'top_k': -1,
 'backend': 'auto',
 'dtype': 'auto',
 'gpu_memory_utilization': 0.85}

In [3]:
import importlib.util
import json

import numpy as np
import torch

from marinfold.document_structures.contacts_v1 import (
    GenerationConfig, InferenceConfig, RawContact, build_document, predict,
    residues_from_sequence, structure_from_sequence)


def resolve_backend(name: str) -> str:
    """vLLM where it is installed and supported (Ampere+), transformers otherwise."""
    if name == "transformers":
        return "transformers"
    usable = (importlib.util.find_spec("vllm") is not None and torch.cuda.is_available()
              and torch.cuda.get_device_capability()[0] >= 8)
    if name == "vllm" and not usable:
        print("note: vLLM asked for but not usable here — using transformers")
    return "vllm" if usable else "transformers"


def resolve_dtype(name: str) -> str:
    """bfloat16 on any GPU. float16 overflows this model's residual stream and dies in sampling."""
    if name != "auto":
        return name
    return "bfloat16" if torch.cuda.is_available() else "float32"


inputs = figlib.Inputs()
targets, ground_truth = figlib.load_legacy_universe(inputs)
dataset_name, stem = PROTEIN.split("__", 1)
target = targets[(targets.dataset == dataset_name) & (targets.stem == stem)].iloc[0]
truth = ground_truth[(dataset_name, stem)]
print(f"{PROTEIN}  L={target.L}  {len(truth['contacts'])} ground-truth contacts  "
      f"{len(truth['resolved'])} resolved residues")

denovo_pdb__1qys_A  L=92  198 ground-truth contacts  92 resolved residues


In [4]:
backend, dtype = resolve_backend(BACKEND), resolve_dtype(DTYPE)
model_identity = figlib.model_identity(MODEL)
print(f"backend {backend} · dtype {dtype} · rope_theta {model_identity['rope_theta']}")

config = InferenceConfig(
    model=MODEL, backend=backend, method="rollout", keep_matrix=True,
    n_rollouts=N_ROLLOUTS, temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
    dtype=dtype, min_seq_separation=figlib.MIN_SEPARATION,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION)

record = next(iter(predict(config, structures=[
    structure_from_sequence(target.input_seq, entry_id=stem)])))
# The band within MIN_SEPARATION comes back as NaN — never a candidate pair, but NaN poisons
# argsort, so it is pushed below every real score.
score = np.nan_to_num(np.asarray(record["score_matrix"], dtype=float), nan=-1e9)
metrics = figlib.score_metrics(score, truth)
headline = metrics[(metrics.range == "all") & (metrics.cut == "R")].iloc[0]
print(f"R-precision {headline.value:.3f} over {int(headline.n_true)} true contacts "
      f"({int(headline.n_candidate)} candidate pairs)")
metrics[metrics.cut.isin(["R", "AUC"])]

/tmp/claude-1000/-home-bizon-git-MarinFold--claude-worktrees-evals-exploration-notebook-cba1c7/a4462f4e-9ca0-491a-ba4b-e1b116817039/scratchpad/nbenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


backend transformers · dtype bfloat16 · rope_theta 500000


`rope_scaling`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=8192


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:  50%|█████     | 1/2 [00:00<00:00,  5.41it/s]

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  9.04it/s]

R-precision 0.697 over 76 true contacts (3741 candidate pairs)


,range,cut,n_candidate,n_true,value
3,all,R,3741,76,0.697368
4,all,AUC,3741,76,0.991046
8,short,R,501,12,0.833333
9,short,AUC,501,12,0.998125
13,medium,R,894,18,0.666667
14,medium,AUC,894,18,0.994165
18,long,R,2346,46,0.652174
19,long,AUC,2346,46,0.987089


In [5]:
# The document itself, so a format panel can be drawn from the real thing rather than a paraphrase.
document = build_document(
    stem, residues_from_sequence(target.input_seq),
    [RawContact(seq_i=int(i), seq_j=int(j), degree=float(d)) for i, j, d in truth["contacts"]],
    config=GenerationConfig())

figlib.write_dataset(
    DATASET,
    notebook="1_make_top7_heatmap_data.ipynb",
    parameters=PARAMETERS,
    inputs=inputs,
    files={
        "score.npy": lambda path: np.save(path, score.astype(np.float32)),
        "ground_truth.json": json.dumps({
            "dataset": dataset_name, "stem": stem, "L": int(truth["L"]),
            "resolved": [int(i) for i in truth["resolved"]],
            "contacts": [[int(i), int(j), float(d)] for i, j, d in truth["contacts"]],
            "sequence": target.input_seq,
        }, indent=2).encode(),
        "metrics.csv": lambda path: metrics.to_csv(path, index=False),
        "document.txt": document.document.encode(),
    },
    extra={
        "model": model_identity,
        "recipe": {"method": "rollout+resample+tiebreak", "backend": backend, "dtype": dtype,
                   "n_rollouts": N_ROLLOUTS, "temperature": TEMPERATURE, "top_p": TOP_P,
                   "top_k": TOP_K, "min_seq_separation": figlib.MIN_SEPARATION},
        "protein": {"dataset": dataset_name, "stem": stem, "L": int(target.L),
                    "sequence_sha256": figlib.digest(target.input_seq.encode()),
                    "n_true_contacts": int(headline.n_true),
                    "n_candidate_pairs": int(headline.n_candidate)},
        "result": {"r_precision_all": float(headline.value),
                   "auc_all": float(metrics[(metrics.range == "all")
                                            & (metrics.cut == "AUC")].value.iloc[0])},
    })

wrote 4 file(s) + metadata.json to /home/bizon/git/MarinFold/.claude/worktrees/evals-exploration-notebook-cba1c7/notebooks/figures/data/1_top7_heatmap
   score.npy                            33,984 B  b578fe74da53
   ground_truth.json                    12,744 B  b63c46f5754f
   metrics.csv                             740 B  27a549997cfc
   document.txt                          2,832 B  c2113d84477d


PosixPath('/home/bizon/git/MarinFold/.claude/worktrees/evals-exploration-notebook-cba1c7/notebooks/figures/data/1_top7_heatmap')